# 00 · ¿Qué mide un VNA?

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/telmomm/nanovna-measurement-course/HEAD?labpath=es%2Fnotebooks%2F00_introduccion_vna.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/telmomm/nanovna-measurement-course/blob/main/es/notebooks/00_introduccion_vna.ipynb)

Ya sabes medir una tensión con un multímetro y observar una señal con un osciloscopio. Un analizador vectorial de redes (VNA, *vector network analyzer*) responde a otra pregunta: *si envío una onda a este dispositivo, ¿cuánto vuelve y con qué retardo?* Este cuaderno explica por qué esa es la pregunta adecuada en radiofrecuencia y qué aspecto tiene la respuesta.

## Objetivos de aprendizaje

Al terminar este cuaderno serás capaz de:

- explicar qué son las ondas incidente y reflejada en un puerto, y por qué aparecen;
- definir los parámetros S de una red de un puerto y de dos puertos;
- calcular cuánta potencia refleja una carga y expresarla como pérdidas de retorno;
- decir qué mide un VNA que no mide un multímetro ni un osciloscopio.

**Necesitas:** fasores e impedancia compleja, y NumPy y Matplotlib básicos. Sin hardware.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/telmomm/nanovna-measurement-course

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import skrf as rf

import vnacourse
from vnacourse import plotting

plotting.use_course_style()

for module in (np, matplotlib, rf, vnacourse):
    print(f"{module.__name__:<12} {module.__version__}")

Si la celda anterior ha mostrado cuatro números de versión, tu entorno está listo para todo el curso.

## 1. Cuando un cable deja de ser un cable

En teoría de circuitos un cable es un nudo: la tensión es la misma en sus dos extremos en cada instante. Es una aproximación. Una señal recorre un cable a una velocidad finita $v$, alrededor del 70 % de la velocidad de la luz en un coaxial típico, de modo que una sinusoide de frecuencia $f$ ocupa una longitud

$$\lambda = \frac{v}{f}$$

a lo largo del cable. Si el cable es mucho más corto que esa longitud de onda, sus dos extremos ven prácticamente la misma tensión y la aproximación es válida. Si no lo es, la tensión depende de *en qué punto* del cable mires.

> **Predice antes de ejecutar.** Tienes un cable de 1 m. ¿A cuál de estas frecuencias deja de ser «solo un cable»: 50 Hz, 1 MHz, 100 MHz, 1 GHz?

In [ ]:
c0 = 299_792_458.0  # velocidad de la luz en el vacío, m/s
v = 0.7 * c0  # velocidad de la onda en un cable coaxial típico
cable_length = 1.0  # m

print(f"{'frecuencia':>10}  {'long. de onda':>13}  {'cable de 1 m, en long. de onda':>31}")
for f in (50, 1e3, 1e6, 100e6, 1e9):
    wavelength = v / f
    print(f"{f:10.3g}  {wavelength:11.3g} m  {cable_length / wavelength:31.2g}")

A 50 Hz la longitud de onda es de miles de kilómetros y el cable no es nada. A 1 MHz sigue siendo menos de una centésima de longitud de onda. A 100 MHz el cable mide media longitud de onda, y a 1 GHz contiene casi cinco. Una regla práctica habitual: en cuanto una conexión es más larga que una décima de longitud de onda, hay que pensar en términos de **ondas**.

Ese es el régimen de un VNA, y también el de buena parte de la instrumentación biomédica que trabaja en radiofrecuencia: implantes inalámbricos, antenas para dispositivos vestibles, bobinas de resonancia magnética, medida dieléctrica de tejidos.

## 2. Dos ondas en una línea

Un cable diseñado para transportar ondas es una **línea de transmisión**. Tiene una **impedancia característica** $Z_0$, casi siempre 50 Ω en instrumentación: la relación entre tensión y corriente de una onda que viaja por ella.

Envía una onda por la línea hacia una carga de impedancia $Z_L$. En la carga, tensión y corriente deben cumplir $V = Z_L I$, pero la onda que llega las trae en la relación $Z_0$. Si $Z_L \neq Z_0$, las dos condiciones no pueden cumplirse con una sola onda, y aparece una segunda que viaja de vuelta: la **onda reflejada**. Su tamaño respecto a la incidente es el **coeficiente de reflexión**:

$$\Gamma = \frac{V^-}{V^+} = \frac{Z_L - Z_0}{Z_L + Z_0}$$

En la línea, las dos ondas se suman. A una distancia $d$ de la carga, la amplitud de la tensión total es

$$|V(d)| = |V^+|\;\bigl|1 + \Gamma\, e^{-j 2 \beta d}\bigr| \qquad \beta = \frac{2\pi}{\lambda}$$

El factor $e^{-j2\beta d}$ es la fase que pierde la onda reflejada al ir hasta la carga y volver.

> **Predice antes de ejecutar.** Representamos la amplitud de la tensión a lo largo de 1 m de línea a 300 MHz para tres cargas: 50 Ω, 150 Ω y un cortocircuito. ¿Con qué carga la amplitud es igual en todas partes? ¿Con cuál se anula en algún punto?

In [ ]:
z0 = 50.0


def reflection(z_load):
    """Coeficiente de reflexión de una carga en una línea de 50 Ω."""
    return (z_load - z0) / (z_load + z0)


f = 300e6
beta = 2 * np.pi * f / v
d = np.linspace(0, 1.0, 500)  # distancia a la carga, m

fig, ax = plt.subplots()
for name, z_load in (("50 Ω", 50.0), ("150 Ω", 150.0), ("cortocircuito", 0.0)):
    gamma = reflection(z_load)
    amplitude = np.abs(1 + gamma * np.exp(-2j * beta * d))
    ax.plot(d, amplitude, label=f"{name}  (Γ = {gamma:+.1f})")
ax.set(xlabel="distancia a la carga (m)", ylabel="amplitud de la tensión, $|V| / |V^+|$")
ax.set_title("Tensión a lo largo de la línea a 300 MHz")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)

print(f"Longitud de onda en el cable a 300 MHz: {v / f:.2f} m")

- **50 Ω** ($\Gamma = 0$): no hay reflexión. La amplitud es la misma en todas partes. La carga está **adaptada**.
- **150 Ω** ($\Gamma = +0.5$): las dos ondas se suman en unos puntos y se cancelan en parte en otros, y forman una **onda estacionaria** entre 0.5 y 1.5.
- **Cortocircuito** ($\Gamma = -1$): todo se refleja, invertido. La amplitud va de 0 al doble de la onda incidente, con ceros cada media longitud de onda.

Imagina que sondeas esta línea con un osciloscopio. La amplitud que lees depende de dónde toques, y en cada punto solo ves la *suma* de las dos ondas. Con una sola lectura no puedes saber cuánto va hacia la carga y cuánto vuelve.

## 3. Ondas en lugar de voltios: $a$, $b$ y $S_{11}$

Como la información la llevan las ondas incidente y reflejada, la ingeniería de RF trabaja directamente con ellas. En un puerto se definen

$$a = \frac{V^+}{\sqrt{Z_0}} \quad\text{(onda incidente)} \qquad b = \frac{V^-}{\sqrt{Z_0}} \quad\text{(onda reflejada)}$$

El escalado por $\sqrt{Z_0}$ hace que $\tfrac12|a|^2$ sea la potencia que viaja hacia el dispositivo y $\tfrac12|b|^2$ la que vuelve. Su cociente es el **parámetro S de reflexión**:

$$S_{11} = \frac{b_1}{a_1}$$

Para una carga sola al final del puerto, $S_{11}$ *es* el coeficiente de reflexión $\Gamma$. Es un número complejo: su módulo dice cuánto vuelve y su fase, con qué retardo.

Dos magnitudes derivadas se usan constantemente:

- la fracción de potencia reflejada, $|S_{11}|^2$;
- las **pérdidas de retorno** en decibelios, $-20\log_{10}|S_{11}|$: cuánto por debajo de la onda incidente está la reflejada. Unas pérdidas de retorno *mayores* indican una *mejor* adaptación.

In [ ]:
print(f"{'carga':>14}  {'Γ':>6}  {'potencia reflejada':>18}  {'pérd. de retorno':>16}")
for name, z_load in (
    ("cortocircuito", 0.0),
    ("25 Ω", 25.0),
    ("50 Ω", 50.0),
    ("75 Ω", 75.0),
    ("150 Ω", 150.0),
):
    gamma = reflection(z_load)
    reflected = abs(gamma) ** 2
    return_loss = "∞" if gamma == 0 else f"{abs(20 * np.log10(abs(gamma))):.1f} dB"
    print(f"{name:>14}  {gamma:+6.2f}  {reflected:17.0%}  {return_loss:>16}")
print(f"{'abierto':>14}  {1:+6.2f}  {1:17.0%}  {'0.0 dB':>16}")

Una carga de 75 Ω en una línea de 50 Ω refleja solo el 4 % de la potencia: la desadaptación es indulgente. Fíjate también en que 25 Ω y 100 Ω darían el mismo $|\Gamma|$ con signos opuestos. El módulo por sí solo no te dice cuál tienes; la fase sí.

## 4. Qué hace el VNA

Un VNA tiene una fuente que recorre una lista de frecuencias. En cada una envía una onda $a_1$ por su puerto, separa la onda $b_1$ que vuelve y mide el cociente $b_1/a_1$ **en módulo y en fase**. Esto último es lo que significa «vectorial» en su nombre.

Simulemos lo que vería. El dispositivo es una resistencia de 75 Ω al final de 20 cm de cable. El cable retrasa la onda $\tau = \ell / v$ en cada sentido, así que la reflexión llega al puerto con un retraso de fase de $2\pi f \cdot 2\tau$:

$$S_{11}(f) = \Gamma_L\, e^{-j 4\pi f \tau}$$

> **Predice antes de ejecutar.** La resistencia refleja la misma fracción de la onda a todas las frecuencias. ¿Cambiará algo en $S_{11}$ con la frecuencia?

In [ ]:
f_hz = np.linspace(50e3, 1.45e9, 401)
f_mhz = f_hz / 1e6

length = 0.20  # m
tau = length / v
a1 = np.ones(f_hz.shape, dtype=complex)  # la onda incidente, nuestra referencia
b1 = reflection(75.0) * np.exp(-4j * np.pi * f_hz * tau) * a1
s11 = b1 / a1

fig, (ax_mag, ax_phase) = plt.subplots(2, 1, figsize=(7.5, 5.2), sharex=True)
ax_mag.plot(f_mhz, np.abs(a1), label="incidente $|a_1|$")
ax_mag.plot(f_mhz, np.abs(b1), label="reflejada $|b_1|$")
ax_mag.set(ylabel="módulo", ylim=(0, 1.2))
ax_mag.set_title("Resistencia de 75 Ω tras 20 cm de cable")
ax_mag.legend(loc="center right")
ax_phase.plot(f_mhz, np.degrees(np.angle(s11)), color=plotting.SERIES[2])
ax_phase.set(
    xlabel="frecuencia (MHz)", ylabel="fase de $S_{11}$ (°)", yticks=[-180, -90, 0, 90, 180]
)

print(f"Retardo del cable en un sentido: {tau * 1e9:.2f} ns")

El módulo es plano, en 0.2. La fase cae de forma continua (los saltos de −180° a +180° son solo el ángulo dando la vuelta) porque el mismo cable mide más longitudes de onda a mayor frecuencia. La pendiente de esa fase es una medida directa del retardo del cable.

Mira ahora lo que mostraría un instrumento que solo mide la *amplitud de la tensión total* en el puerto: $|a_1 + b_1|$.

In [ ]:
total = np.abs(a1 + b1)

fig, ax = plt.subplots()
ax.plot(f_mhz, total, color=plotting.SERIES[1])
ax.set(xlabel="frecuencia (MHz)", ylabel="$|a_1 + b_1|$", ylim=(0, 1.4))
ax.set_title("Amplitud de la tensión total en el puerto");

La tensión total oscila entre 0.8 y 1.2 con la frecuencia. Solo con esta curva no podrías decir si el dispositivo cambia con la frecuencia o si estás viendo una reflexión constante a través de un cable. El VNA sí puede, porque mantiene separadas las dos ondas y conserva la fase.

### Lo mismo con scikit-rf

[scikit-rf](https://scikit-rf.org) es la biblioteca de Python que usa este curso para manejar parámetros S. Su objeto central es `Network`: un eje de frecuencias más los parámetros S en cada frecuencia. Construimos el cable como una red de dos puertos y la resistencia como una red de un puerto, las conectamos con `**` y comparamos con nuestra fórmula.

In [ ]:
frequency = rf.Frequency.from_f(f_hz, unit="Hz")

s_cable = np.zeros((len(f_hz), 2, 2), dtype=complex)
s_cable[:, 1, 0] = s_cable[:, 0, 1] = np.exp(-2j * np.pi * f_hz * tau)
cable = rf.Network(frequency=frequency, s=s_cable, z0=50, name="cable de 20 cm")

resistor = rf.Network(
    frequency=frequency, s=np.full(len(f_hz), reflection(75.0)), z0=50, name="75 Ω"
)

device = cable**resistor
device.name = "75 Ω tras 20 cm de cable"
print(device)
print("¿El mismo S11 que nuestra fórmula?", np.allclose(device.s[:, 0, 0], s11))

`device.s` tiene forma (frecuencias, puertos, puertos), así que `device.s[:, 0, 0]` es $S_{11}$ a cada frecuencia. También te encontrarás `device.f` (frecuencias en Hz), `device.s_db` y `device.s_deg`.

## 5. Dos puertos

El cable anterior tiene dos extremos, así que tiene dos puertos y cuatro parámetros S. Cada uno es «onda que sale por el puerto $i$ dividida por la onda que entra por el puerto $j$, con el otro puerto adaptado»:

$$
\begin{bmatrix} b_1 \\ b_2 \end{bmatrix} =
\begin{bmatrix} S_{11} & S_{12} \\ S_{21} & S_{22} \end{bmatrix}
\begin{bmatrix} a_1 \\ a_2 \end{bmatrix}
$$

- $S_{11}$, $S_{22}$: **reflexión** en cada puerto.
- $S_{21}$, $S_{12}$: **transmisión** del puerto 1 al 2 y de vuelta.

In [ ]:
k = 200  # un punto de frecuencia
print(f"Matriz S del cable a {f_mhz[k]:.0f} MHz:")
print(np.round(cable.s[k], 3))
print(f"|S21| = {abs(cable.s[k, 1, 0]):.2f}, fase {np.degrees(np.angle(cable.s[k, 1, 0])):.0f}°")

Nuestro cable ideal no refleja nada ($S_{11} = S_{22} = 0$) y lo transmite todo ($|S_{21}| = 1$) con un retardo de fase. Un filtro tendría $|S_{21}| \approx 1$ en su banda de paso y $\approx 0$ fuera de ella; un amplificador tendría $|S_{21}| > 1$.

Este curso trata de medidas de **un puerto**: solo $S_{11}$. Es el caso más sencillo, es lo que necesitas para medir una impedancia, y todas las ideas sobre errores y calibración se trasladan a dos puertos.

## 6. Tres instrumentos

| | Multímetro | Osciloscopio | VNA |
|---|---|---|---|
| Mide | tensión, corriente, resistencia | tensión frente al tiempo, en un punto | cociente de ondas, frente a la frecuencia |
| Estímulo | continua o baja frecuencia | ninguno (observa) | su propia sinusoide de barrido |
| Resultado | un número | una forma de onda | un número complejo por frecuencia |
| Separa la onda incidente de la reflejada | no | no: ve su suma | **sí** |
| Conserva la fase | no | respecto a su disparo | **sí**, respecto a su propia fuente |
| Te dice | «esto son 75 Ω» en continua | «la señal es así, aquí» | «así responde el dispositivo a una onda, a cada frecuencia» |

Un VNA es el equivalente en RF del óhmetro: caracteriza un *dispositivo*, no una señal. El NanoVNA que se usa en este curso es uno de bolsillo y de bajo coste. Como verás a partir del cuaderno 02, lo que muestra todavía no es la verdad sobre el dispositivo.

## Comprueba lo que has entendido

1. Una carga refleja el 25 % de la potencia incidente. ¿Cuánto vale $|\Gamma|$? ¿Y las pérdidas de retorno?

<details><summary>Respuesta</summary>

$|\Gamma| = \sqrt{0.25} = 0.5$. Pérdidas de retorno $= -20\log_{10} 0.5 = 6$ dB.
</details>

2. En la sección 2, los ceros de la onda estacionaria del cortocircuito están separados 0.35 m a 300 MHz. ¿Por qué esa distancia?

<details><summary>Respuesta</summary>

El patrón depende de $e^{-j2\beta d}$, que se repite cuando $2\beta d$ aumenta $2\pi$, es decir, cada $d = \lambda/2$. La longitud de onda en el cable es 0.70 m, así que los ceros están a 0.35 m.
</details>

3. Mides $|S_{11}| = 0.2$. ¿Puedes saber si la carga es de 75 Ω o de 33 Ω?

<details><summary>Respuesta</summary>

No con el módulo: 75 Ω da $\Gamma = +0.2$ y 33.3 Ω da $\Gamma = -0.2$. La fase (0° o 180°) las distingue, siempre que no haya delante una longitud de cable desconocida, que giraría la fase.
</details>

4. ¿Por qué un osciloscopio no puede sustituir a un VNA para medir una reflexión?

<details><summary>Respuesta</summary>

En un solo punto ve la suma de las ondas incidente y reflejada. Separarlas requiere hardware direccional (un acoplador o un puente, como dentro de un VNA) o medidas en varias posiciones conocidas.
</details>

## Ejercicios

Las soluciones resueltas están en [`exercises/solutions/00_solutions.ipynb`](../exercises/solutions/00_solutions.ipynb). Inténtalos antes.

1. **¿Cuándo es corto un cable?** Para un cable de 10 cm con $v = 0.7c$, halla la frecuencia a la que mide una décima de longitud de onda. Repite para la patilla de 1 cm de un componente. ¿Qué te dice esto sobre medir a 1 GHz?
2. **Circuito abierto.** Añade un circuito abierto ($Z_L \to \infty$, $\Gamma = +1$) a la gráfica de onda estacionaria de la sección 2. ¿Dónde están sus ceros en comparación con los del cortocircuito?
3. **De pérdidas de retorno a impedancia.** La hoja de datos de una antena promete unas pérdidas de retorno mejores que 10 dB. ¿Cuál es el mayor $|\Gamma|$ permitido, qué fracción de la potencia puede reflejarse y entre qué dos valores puramente resistivos puede estar su impedancia?
4. **Un cable más largo.** En la sección 4, cambia el cable de 20 cm por uno de 1 m. ¿Cómo cambia la gráfica de fase? Estima el retardo del cable a partir de la pendiente de la fase desenrollada (`np.unwrap`) y compáralo con $\ell / v$.

In [ ]:
# Tu trabajo aquí

## Resumen

- Cuando una conexión no es corta comparada con la longitud de onda, la tensión depende de la posición y hay que pensar en ondas.
- Una carga distinta de la $Z_0$ de la línea refleja parte de la onda: $\Gamma = (Z_L - Z_0)/(Z_L + Z_0)$.
- $S_{11} = b_1/a_1$ es el cociente entre la onda reflejada y la incidente en un puerto: un número complejo por frecuencia. $|S_{11}|^2$ es la fracción de potencia reflejada.
- Un VNA genera la onda incidente, separa la reflejada y mide su cociente en módulo y fase a lo largo de la frecuencia.
- La fase lleva información que el módulo no tiene: el signo de la desadaptación y cualquier retardo delante del dispositivo.

**Siguiente:** [01 · Coeficiente de reflexión y carta de Smith](01_reflexion_smith.ipynb) convierte $\Gamma$ en impedancia y viceversa, y presenta la carta en la que los ingenieros de RF leen ambas a la vez.